
# Learning about the Gymnasium Library

In this notebook, that was created based on the [GitHub](https://github.com/PacktPublishing/Deep-Reinforcement-Learning-Hands-On-Third-Edition) of the Third Edition [Deep Reinforcement Learning Hands-On](https://www.oreilly.com/library/view/deep-reinforcement-learning/9781835882702/) book, you will learn the basics of [Gymnasium](https://gymnasium.farama.org/), a library used to provide a uniform API for RL agents and lots of RL environments.

You will try a first randomly behaving agent and become more familiar with the basic concepts of RL, in order to have an understanding of:
- The high-level requirements that need to be implemented to plug the agent into the RL framework
- A basic, pure-Python implementation of the random RL agent
- The Gymnasium library and its implementation

To render and show video with gym.wrappers.RecordVideo in Google Colab, it is required to install necessary libraries.

In [25]:
%%capture
#!pip install pyvirtualdisplay
#!apt-get install -y xvfb python-opengl ffmpeg

Let's first import a few dependencies we'll need.




In [26]:
import random
from typing import List

**Environment class**: Everything that is external to the agent and has the responsibility of providing observations and giving rewards. The environment changes its state based on the agent’s actions.

In [27]:
class Environment:
    def __init__(self):
        self.steps_left = 10

    def get_observation(self) -> List[float]:
        return [0.0, 0.0, 0.0]

    def get_actions(self) -> List[int]:
        return [0, 1]

    def is_done(self) -> bool:
        return self.steps_left == 0

    def action(self, action: int) -> float:
        if self.is_done():
            raise Exception("Game is over")
        self.steps_left -= 1
        return random.random()

**Agent class**: A thing, or person, that takes an active role. In practice, the agent is some piece of code that implements some policy. Basically, this policy decides what action is needed at every time step, given our observations.

In [28]:
class Agent:
    def __init__(self):
        self.total_reward = 0.0

    def step(self, env: Environment):
        current_observation = env.get_observation()
        actions = env.get_actions()
        reward = env.action(random.choice(actions))
        self.total_reward += reward

First execution with a random agent

In [29]:
# Create an environment
env = Environment()

# Create an agent
agent = Agent()

# Perform steps
while not env.is_done():
    agent.step(env)

print("Total reward got: %.4f" % agent.total_reward)

Total reward got: 3.9715


Creating the [Cart Pole](https://gymnasium.farama.org/environments/classic_control/cart_pole/) environment. A pole is attached by an un-actuated joint to a cart, which moves along a frictionless track. The pendulum is placed upright on the cart and the goal is to balance the pole by applying forces in the left and right direction on the cart.

In [30]:
import gymnasium as gym

# Create a Gym environment
env = gym.make("CartPole-v1")
total_reward = 0.0
total_steps = 0

# Reset the environment
observation, info = env.reset()

# Perform steps and observe the reward
while True:
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    total_steps += 1
    if terminated or truncated:
        break

print("Episode done in %d steps, total reward %.2f" % (total_steps, total_reward))

Episode done in 15 steps, total reward 15.00


**RandomActionWrapper class**: Used to issue the random actions, making the agent to explore the environment and from time to time drift away from the beaten track of its policy.

In [31]:
class RandomActionWrapper(gym.ActionWrapper):
    def __init__(self, env: gym.Env, epsilon: float = 0.1):
        super(RandomActionWrapper, self).__init__(env)
        self.epsilon = epsilon

    def action(self, action: gym.core.WrapperActType) -> gym.core.WrapperActType:
        if random.random() < self.epsilon:
            action = self.env.action_space.sample()
            print(f"Random action {action}")
            return action
        return action

In [32]:
# Create a Gym environment with RandomActionWrapper
env = RandomActionWrapper(gym.make("CartPole-v1"))

# Reset the wrapped environment
observation, info = env.reset()
total_reward = 0.0

# Perform steps and observe the reward
while True:
    observation, reward, terminated, truncated, info = env.step(0)
    total_reward += reward
    if terminated or truncated:
        break

print(f"Reward got: {total_reward:.2f}")

Random action 1
Random action 1
Reward got: 11.00


In [48]:
# Create a Gym environment that supports rgb_array rendering
env = gym.make("CartPole-v1", render_mode="rgb_array")

# Wrap the environment with RecordVideo to save videos
wrapped_env = gym.wrappers.RecordVideo(env, video_folder="./videos")

total_reward = 0.0
total_steps = 0

# Reset the wrapped environment, which will open the rendering window
observation, info = wrapped_env.reset()

# Perform steps and observe the rendering
while True:
    action = wrapped_env.action_space.sample()
    observation, reward, terminated, truncated, info = wrapped_env.step(action)
    total_reward += reward
    total_steps += 1
    if terminated or truncated:
        break

print(f"Episode done in {total_steps} steps, total reward {total_reward:.2f}")

# Close the environment to finalize video saving
wrapped_env.close()

Episode done in 36 steps, total reward 36.00


In [34]:
# from pyvirtualdisplay import Display

# # Virtual display
# display = Display(visible=0, size=(1400, 900))
# display.start()

In [35]:
from base64 import b64encode
from IPython.display import HTML

# Find the path to the recorded video (assuming one was recorded)
video_path = "./videos/rl-video-episode-0.mp4" # Adjust episode number if needed

# Gets a string containing a b4-encoded version of the MP4 video at the specified path
def show_video(video_path):
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f'<video width="400" controls><source src="{data_url}" type="video/mp4"></video>')

In [36]:
show_video(video_path)

---